# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1.1 Two grains, kept deliberately separate

| | Source grain (what the table stores) | Analysis grain (what one training row is) |
|---|---|---|
| Warehouse | `fact_content_daily_performance` = one row per **`report_date` x `client_hash_id` x `content_hash_id`** | one row per **`content_hash_id` observed at a single decision date `T`** |
| Starter CSV | `content_refresh_anonymized.csv` = one row per **`content_id`**, a single trailing-90-day snapshot | same row, but there is no future window to score against |

The daily table is a fact table. Collapsing it to one row per page without first fixing a decision
date is the mistake that makes every later number unreadable, so the decision date is defined here
and every later notebook reads it from the contract this notebook writes.

### 1.2 The decision point and the two windows

For a decision date `T`, one analysis row is one content item, with:

- **Feature window** `W_f = [T - 89, T]` — 90 days, everything the model is allowed to see.
- **Label window** `W_l = [T + 1, T + 30]` — 30 days, strictly after `T`, used only to build the outcome.

Nothing measured inside `W_l` may ever become a feature. That single rule is what makes the label a
future observed outcome rather than a restatement of the present, and it is the rule the leakage
checks in section 3 exist to enforce.

**Development decision date: `T = 2026-03-31`.** Chosen mid-panel on purpose. The dataset's final
month (June 2026) is the natural outcome window for any past-to-future label, so developing there
would mean tuning inside the test window. June 2026 is therefore sealed and untouched until final
evaluation.

| Setting | Value | Why |
|---|---|---|
| `DEV_DECISION_DATE` | `2026-03-31` | mid-panel, leaves a full clean 30-day label window in April |
| `FEATURE_WINDOW` | `2026-01-01` to `2026-03-31` | 90 days ending at `T` |
| `LABEL_WINDOW` | `2026-04-01` to `2026-04-30` | 30 days strictly after `T` |
| `SEALED_TEST_MONTH` | `2026-06` | final month, reserved for final evaluation only |
| `ITERATION_PARTITION` | `month=2026-03` | iterate on one partition, never repeated 79M-row scans |

### 1.3 Windows are per client, not per calendar

The panel is unbalanced: each client's history begins whenever their tracking started, and rows
before a client's GA4 start carry `ga4_data_available = FALSE` with GA4 columns zero-filled. A
global calendar window would silently read "not tracked yet" as "no traffic". Any client whose
`gsc_data_start` is later than `T - 89` is therefore excluded from that decision date rather than
padded with zeros.


In [1]:
# Section 1 - unit of analysis and time windows, stated as executable settings.
import json
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

# --- Contract settings every later notebook will read back -------------------
DEV_DECISION_DATE = date(2026, 3, 31)
FEATURE_WINDOW_DAYS = 90
LABEL_WINDOW_DAYS = 30
FEATURE_WINDOW = (DEV_DECISION_DATE - timedelta(days=FEATURE_WINDOW_DAYS - 1), DEV_DECISION_DATE)
LABEL_WINDOW = (DEV_DECISION_DATE + timedelta(days=1), DEV_DECISION_DATE + timedelta(days=LABEL_WINDOW_DAYS))
SEALED_TEST_MONTH = "2026-06"
ITERATION_PARTITION = "month=2026-03"

print("ANALYSIS GRAIN : one content_hash_id observed at a single decision date T")
print(f"T              : {DEV_DECISION_DATE}")
print(f"feature window : {FEATURE_WINDOW[0]} -> {FEATURE_WINDOW[1]}  ({FEATURE_WINDOW_DAYS} days)")
print(f"label window   : {LABEL_WINDOW[0]} -> {LABEL_WINDOW[1]}  ({LABEL_WINDOW_DAYS} days, strictly after T)")
print(f"sealed test    : {SEALED_TEST_MONTH} (never used for development)")
print(f"iterate on     : {ITERATION_PARTITION}")

# The windows must not touch. This is the whole leakage guarantee in one assert.
assert LABEL_WINDOW[0] > FEATURE_WINDOW[1], "label window must start strictly after the decision date"
print("\n[OK] feature and label windows do not overlap")

# --- Starter CSV: verify the claimed grain rather than assuming it -----------
def find_starter_csv() -> Path:
    """Resolve the starter CSV from Colab, repo root, or notebook-relative runs."""
    for candidate in (
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
        Path("/content/FlyRank-Machine-Learning-Internship/data/raw/content_refresh_anonymized.csv"),
    ):
        if candidate.exists():
            return candidate
    raise FileNotFoundError("starter CSV not found; check SETUP.md for the data path")

starter_path = find_starter_csv()
starter = pd.read_csv(starter_path)

dupes = starter.groupby("content_id").size()
dupes = dupes[dupes > 1]

print(f"\nSTARTER CSV    : {starter_path}")
print(f"  rows         : {len(starter):,}")
print(f"  columns      : {len(starter.columns)}")
print(f"  clients      : {starter['client_id'].nunique()}")
print(f"  grain probe  : {len(dupes)} content_id values appear more than once")
assert len(dupes) == 0, "starter grain is NOT one row per content_id"
print("  [OK] starter grain holds: one row = one content item")
print("  [NOTE] single trailing-90d snapshot: no future window exists here,")
print("         so the starter CSV can prototype features but cannot carry the forward label.")


ANALYSIS GRAIN : one content_hash_id observed at a single decision date T
T              : 2026-03-31
feature window : 2026-01-01 -> 2026-03-31  (90 days)
label window   : 2026-04-01 -> 2026-04-30  (30 days, strictly after T)
sealed test    : 2026-06 (never used for development)
iterate on     : month=2026-03

[OK] feature and label windows do not overlap

STARTER CSV    : ..\..\data\raw\content_refresh_anonymized.csv
  rows         : 30,000
  columns      : 44
  clients      : 32
  grain probe  : 0 content_id values appear more than once
  [OK] starter grain holds: one row = one content item
  [NOTE] single trailing-90d snapshot: no future window exists here,
         so the starter CSV can prototype features but cannot carry the forward label.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Two datasets, two manifests. All 44 starter columns and all 66 warehouse columns (9 + 26 + 31) are
assigned, and the code cell asserts both manifests are exhaustive against the live schemas, so a
column cannot be silently forgotten and I cannot name a column that does not exist.

### The rule that makes warehouse buckets different: membership depends on the window

On the starter CSV a column has one bucket. On the warehouse the *same column* is a feature or a
label depending on which window it was measured in:

| Column | Measured over `W_f` = [T-89, T] | Measured over `W_l` = [T+1, T+30] |
|---|---|---|
| `ga4_sessions` | **feature** (observed demand before the decision) | **label source** (the forward outcome) |
| `sessions_ai` | **feature** (current AI visibility) | **label source** (secondary AI objective) |

So a bucket is never a property of a column name alone, it is a property of (column, window). Every
aggregate built later must carry its window in its own name, e.g. `ga4_sessions_wf_sum` versus
`ga4_sessions_wl_sum`, so the two can never be confused in a join.

### Feature — knowable before `T`

- **Starter:** demand and competition, content shape, age and freshness, observed performance inside
  the trailing window, and derived rates.
- **`dim_content`:** `keyword_char_count`, `keyword_token_count`, `url_char_count`, `content_type`,
  `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent`, `backlinks`,
  `category_count`, `char_count`, `word_count`, and `content_created_date` / `keyword_created_date`
  (as age at `T`, never as a raw date).
- **`fact_content_daily_performance`, aggregated over `W_f` only:** `gsc_impressions`, `gsc_clicks`,
  `gsc_sum_position`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`,
  `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`,
  `sessions_referral`, `sessions_social`, `sessions_paid`, `scroll_events`, and the AI columns
  `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,
  `ai_other`.

The per-engine AI columns are the reason this lane is worth doing on the warehouse rather than the
starter slice: the starter has only a single `ai_sessions_90d` total, so engine mix is invisible there.

### Label / proxy — the thing predicted, never a feature in the same window

- **Primary:** `ga4_sessions` summed over `W_l` falls below 80% of the `W_f` daily average, computed
  on rows where `ga4_data_available` is true.
- **Secondary:** the AI visibility gap, `sessions_ai` over `W_l` on items with demand in `W_f`.
- **Starter proxy:** `trend_direction` and `trend_pct`. `trend_direction` is computed from
  `trend_pct`, so `trend_pct` is the label in continuous form. Section 3 measures this rather than
  asserting it.

### Context — grouping, joining, splitting, filtering. Never learned from

`content_id`, `client_id` (starter); `content_hash_id`, `client_hash_id`, `keyword_hash_id`,
`url_hash_id`, `report_date`, `month` (warehouse). Plus every eligibility field: `is_active`,
`has_gsc_access`, `has_ga4_access`, `access_profile`, `gsc_data_start`, `ga4_data_start`,
`client_created_date`, `client_updated_date`, `client_has_gsc`, `client_has_ga4`,
`gsc_data_available`, `ga4_data_available`, `is_published`, `is_deleted`.

These decide which rows exist, not what the model learns. `client_hash_id` does real work as the
grouping key for client-held-out splits, because pages from one client share templates and traffic
patterns and would leak straight across a random row split.

### Excluded — with a why for each

| Field | Why excluded |
|---|---|
| `provider_used`, `model_used` | Which vendor enriched the row. A pipeline artefact, not a property of the page; learning from it models FlyRank's tooling history. |
| `last_optimized_date`, `optimization_eligible_date` | These record FlyRank's own optimisation decisions. A model fed them learns to reproduce the existing prioritisation, which is the circular-result trap. |
| `content_updated_date` | Can post-date `T`. Usable only as "days since update, as at `T`", never raw. |
| `trend_direction`, `trend_pct` | The starter label. Listed here too so no later notebook reintroduces them as features. |
| FlyRank product flags (`health_score`, `priority_score`, `action_type`, `refresh_tier`) | Deliberately not shipped. If ever rebuilt, they are a baseline to beat, never a feature and never a label. |


In [2]:
# Section 2 - field manifests for both datasets, asserted exhaustive.
STARTER_BUCKETS = {
    "context": ["content_id", "client_id"],
    "label": ["trend_direction", "trend_pct"],
    "excluded": ["provider_used", "model_used"],
    "feature": [
        "search_volume", "competition", "competition_level", "cpc",
        "content_type", "main_intent", "word_count", "char_count",
        "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
        "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
        "days_with_impressions", "days_with_sessions",
        "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
        "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
        "content_age_days", "age_tier", "age_tier_order", "days_since_last_update",
        "freshness_tier", "word_count_tier", "char_count_tier",
        "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
        "impression_tier", "position_tier",
    ],
}

# Warehouse buckets. "feature" here means "safe once aggregated over W_f only".
WAREHOUSE_BUCKETS = {
    "dim_clients": {
        "context": [
            "client_hash_id", "is_active", "has_gsc_access", "has_ga4_access",
            "access_profile", "client_created_date", "client_updated_date",
            "gsc_data_start", "ga4_data_start",
        ],
        "feature": [], "label": [], "excluded": [],
    },
    "dim_content": {
        "context": ["client_hash_id", "content_hash_id", "keyword_hash_id", "url_hash_id",
                    "is_published", "is_deleted"],
        "feature": ["keyword_char_count", "keyword_token_count", "url_char_count",
                    "content_type", "search_volume", "competition", "competition_level",
                    "cpc", "main_intent", "backlinks", "category_count",
                    "char_count", "word_count", "content_created_date", "keyword_created_date"],
        "label": [],
        "excluded": ["provider_used", "model_used", "last_optimized_date",
                     "optimization_eligible_date", "content_updated_date"],
    },
    "fact_content_daily_performance": {
        "context": ["report_date", "client_hash_id", "content_hash_id", "month",
                    "client_has_gsc", "client_has_ga4",
                    "gsc_data_available", "ga4_data_available"],
        "feature": ["gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_avg_position",
                    "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions",
                    "ga4_total_engagement_sec", "sessions_organic", "sessions_direct",
                    "sessions_referral", "sessions_social", "sessions_paid", "scroll_events",
                    "sessions_ai", "ai_chatgpt", "ai_perplexity", "ai_gemini",
                    "ai_copilot", "ai_claude", "ai_meta", "ai_other"],
        "label": [],
        "excluded": [],
    },
}

# Columns that are a feature in W_f and a label source in W_l. Window decides, not the name.
WINDOW_DEPENDENT = ["ga4_sessions", "sessions_ai", "ai_chatgpt", "ai_perplexity",
                    "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other"]

def check_manifest(name, buckets, actual_cols):
    """A manifest must cover every real column exactly once, and invent none."""
    assigned = [c for cols in buckets.values() for c in cols]
    missing = sorted(set(actual_cols) - set(assigned))
    unknown = sorted(set(assigned) - set(actual_cols))
    dupes = sorted({c for c in assigned if assigned.count(c) > 1})
    status = "OK" if not (missing or unknown or dupes) else "FAIL"
    print(f"  {name:34s} {len(assigned):3d}/{len(actual_cols):3d} assigned  [{status}]")
    if missing: print(f"      unassigned : {missing}")
    if unknown: print(f"      not in data: {unknown}")
    if dupes:   print(f"      duplicated : {dupes}")
    assert not (missing or unknown or dupes), f"{name} manifest is not exhaustive"

print("MANIFEST CHECK - starter")
check_manifest("content_refresh_anonymized.csv", STARTER_BUCKETS, list(starter.columns))

for bucket, cols in STARTER_BUCKETS.items():
    print(f"    {bucket:9s}: {len(cols):2d} fields")
print(f"\n  window-dependent warehouse columns (feature in W_f, label source in W_l): {len(WINDOW_DEPENDENT)}")
print(f"    {', '.join(WINDOW_DEPENDENT)}")
print("\n  Warehouse manifests are asserted against the live schema in section 3.")


MANIFEST CHECK - starter
  content_refresh_anonymized.csv      44/ 44 assigned  [OK]
    context  :  2 fields
    label    :  2 fields
    excluded :  2 fields
    feature  : 38 fields

  window-dependent warehouse columns (feature in W_f, label source in W_l): 9
    ga4_sessions, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other

  Warehouse manifests are asserted against the live schema in section 3.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Starter checks run from the local CSV. Warehouse checks run against the gated release and are the
ones that matter most, because that is where the capstone data comes from. The leakage test is the
core absorbed from the retired ML-05 card, so it lives here rather than in a separate notebook.

1. **Grain** — `content_id` unique on the starter (section 1); `report_date + client_hash_id +
   content_hash_id` unique on the warehouse fact table, probed on a real partition.
2. **Rate columns are x100 percentages**, per `docs/data-dictionary.md`: `ctr = 0.76` means 0.76%,
   not 76%. The check prints each rate column's observed range and counts rows above 100, which is
   where `scroll_rate` and `ai_traffic_pct` legitimately land.
3. **Sentinel values that are not measurements.** `avg_position = 0` means "no data", not rank zero.
4. **Missingness follows `content_type`.** Patterned, not random, so a blind `fillna(0)` injects a
   category signal. Fix is a `has_*` indicator plus imputation, never a bare zero.
5. **Leakage test.** Each candidate feature is scored on how well it alone separates the label.
   Anything near-perfect is the label in disguise. `trend_pct` should fail loudly.
6. **Warehouse schema assertion.** The manifests written in section 2 are checked against the live
   `DESCRIBE` output of all three tables. If FlyRank adds or renames a column, this fails rather
   than silently dropping it.

The warehouse cells run only when `HF_TOKEN` is present. Without it they print the values recorded
on the run of record and say so, so the notebook still executes top to bottom either way. The token
is read from the environment (Colab Secrets or a local variable) and never appears in this notebook,
because this repository is public.


In [3]:
# Section 3 - checks behind each claim. Starter first, warehouse second.
import os

print("=" * 74)
print("CHECK 2 - rate columns are x100 percentages; two exceed 100 by design")
print("=" * 74)
for col in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]:
    print(f"  {col:16s} min={starter[col].min():8.2f}  max={starter[col].max():10.2f}  "
          f"rows over 100 = {int((starter[col] > 100).sum()):5d}")
print("  scroll_rate and ai_traffic_pct over 100 is expected: numerator and denominator")
print("  come from different measurement systems. Not a bug.")

print("\n" + "=" * 74)
print("CHECK 3 - avg_position == 0 is a 'no data' sentinel, not rank zero")
print("=" * 74)
sentinel = int((starter["avg_position"] == 0).sum())
real_pos = starter.loc[starter["avg_position"] > 0, "avg_position"]
print(f"  rows with avg_position == 0 : {sentinel:,} ({sentinel / len(starter) * 100:.2f}%)")
print(f"  mean including sentinels    : {starter['avg_position'].mean():.2f}   <- misleading")
print(f"  mean excluding sentinels    : {real_pos.mean():.2f}   <- the real one")

print("\n" + "=" * 74)
print("CHECK 4 - missingness is patterned by content_type, not random")
print("=" * 74)
watch = ["search_volume", "competition", "cpc", "word_count"]
print(starter.groupby("content_type")[watch].apply(lambda g: g.isna().mean() * 100).round(1).to_string())
print("\n  A blind fillna(0) here encodes content_type into every filled column.")
print("  Contract rule: add has_<col> indicators, then impute; never a bare zero.")

print("\n" + "=" * 74)
print("CHECK 5 - leakage test (core absorbed from the retired ML-05 card)")
print("=" * 74)
label = (starter["trend_direction"] == "down").astype(int)
print(f"  proxy label 'is_declining' positive rate: {label.mean() * 100:.2f}%")

def separation(values, target):
    """|AUC - 0.5| * 2 via ranks: 0.0 = no signal, 1.0 = perfect separation."""
    ok = values.notna()
    if ok.sum() < 100 or values[ok].nunique() < 2 or target[ok].nunique() < 2:
        return float("nan")
    ranked, tgt = values[ok].rank(), target[ok]
    pos, neg = ranked[tgt == 1], ranked[tgt == 0]
    return abs(((pos.mean() - (len(pos) + 1) / 2) / len(neg)) - 0.5) * 2

numeric_cols = [c for c in starter.columns
                if pd.api.types.is_numeric_dtype(starter[c]) and c != "age_tier_order"]
scores = (pd.Series({c: separation(starter[c], label) for c in numeric_cols})
          .dropna().sort_values(ascending=False))

print("\n  Top separators (1.00 = this field alone perfectly predicts the label):")
for col, score in scores.head(8).items():
    verdict = "LEAK - it IS the label" if score > 0.95 else ("suspicious" if score > 0.80 else "plausible signal")
    print(f"    {col:24s} {score:5.3f}   {verdict}")

leaks = sorted(scores[scores > 0.95].index)
print(f"\n  Flagged as leakage: {leaks}")
assert "trend_pct" in leaks, "expected trend_pct to be caught as the label in continuous form"
assert not (set(leaks) & set(STARTER_BUCKETS["feature"])), "a flagged leak is still bucketed as a feature"
print("  [OK] every flagged leak is already excluded from the feature bucket")

# --- Warehouse -------------------------------------------------------------
print("\n" + "=" * 74)
print("CHECK 6 - warehouse schema, grain, and windows (gated release)")
print("=" * 74)

BASE = "hf://datasets/FlyRank/internship-warehouse"
ITER_PARTITION_FILE = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

# Values from the run of record, 2026-08-19, so this cell is honest without a token.
RECORDED = {
    "dim_clients_rows": 104,
    "dim_content_rows": 519_606,
    "march_rows": 9_841_378,
    "march_dates": ("2026-03-01", "2026-03-31"),
    "march_clients": 55,
    "march_content": 331_437,
    "grain_duplicates": 0,
}

warehouse_live = False
try:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("HF_TOKEN not set in this environment")
    import duckdb
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("CREATE SECRET (TYPE HUGGINGFACE, PROVIDER credential_chain);")
    # Cosmetic only, and it raises under Jupyter when ipywidgets is absent.
    # It must never be the reason the real checks below get skipped.
    try:
        con.execute("SET enable_progress_bar=false;")
    except Exception:
        pass
    warehouse_live = True
except Exception as exc:
    print(f"  [SKIPPED LIVE QUERIES] {type(exc).__name__}: {exc}")
    print("  Cause is printed above verbatim: no HF_TOKEN, no network, or no gate access.")
    print("  Falling back to the values recorded on the run of record (2026-08-19).")

if warehouse_live:
    live_schema = {}
    for table, path in [
        ("dim_clients", f"{BASE}/dim_clients.parquet"),
        ("dim_content", f"{BASE}/dim_content.parquet"),
        ("fact_content_daily_performance", ITER_PARTITION_FILE),
    ]:
        live_schema[table] = [r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM '{path}' LIMIT 0").fetchall()]

    print("  manifest vs live schema:")
    for table, buckets in WAREHOUSE_BUCKETS.items():
        check_manifest(table, buckets, live_schema[table])

    n, d0, d1, nc, nk = con.execute(
        "SELECT COUNT(*), MIN(report_date), MAX(report_date), "
        "COUNT(DISTINCT client_hash_id), COUNT(DISTINCT content_hash_id) "
        f"FROM '{ITER_PARTITION_FILE}'").fetchone()
    dup = con.execute(
        "SELECT COUNT(*) FROM (SELECT report_date, client_hash_id, content_hash_id "
        f"FROM '{ITER_PARTITION_FILE}' GROUP BY 1,2,3 HAVING COUNT(*)>=2)").fetchone()[0]
    print(f"\n  iteration partition month=2026-03")
    print(f"    rows     : {n:,}")
    print(f"    dates    : {d0} to {d1}")
    print(f"    clients  : {nc}   content items: {nk:,}")
    print(f"    grain probe (report_date + client + content duplicated): {dup}")
    assert dup == 0, "warehouse grain is not report_date x client_hash_id x content_hash_id"
    print("    [OK] warehouse grain holds")
else:
    for key, value in RECORDED.items():
        print(f"    {key:20s}: {value}")
    print("    [OK on the run of record] grain probe returned 0 duplicates")


CHECK 2 - rate columns are x100 percentages; two exceed 100 by design
  ctr              min=    0.00  max=    100.00  rows over 100 =     0
  engagement_rate  min=    0.00  max=    100.00  rows over 100 =     0
  scroll_rate      min=    0.00  max=    300.00  rows over 100 =   119
  ai_traffic_pct   min=    0.00  max=    300.00  rows over 100 =    23
  scroll_rate and ai_traffic_pct over 100 is expected: numerator and denominator
  come from different measurement systems. Not a bug.

CHECK 3 - avg_position == 0 is a 'no data' sentinel, not rank zero
  rows with avg_position == 0 : 1,205 (4.02%)
  mean including sentinels    : 16.34   <- misleading
  mean excluding sentinels    : 17.03   <- the real one

CHECK 4 - missingness is patterned by content_type, not random
                    search_volume  competition    cpc  word_count
content_type                                                     
comparison article            0.0          0.0    0.0         0.0
feedly article           


  Top separators (1.00 = this field alone perfectly predicts the label):
    trend_pct                1.000   LEAK - it IS the label
    impressions_prev_30d     0.243   plausible signal
    content_age_days         0.183   plausible signal
    impressions_90d          0.169   plausible signal
    days_with_impressions    0.159   plausible signal
    search_volume            0.122   plausible signal
    word_count               0.093   plausible signal
    sessions_prev_30d        0.085   plausible signal

  Flagged as leakage: ['trend_pct']
  [OK] every flagged leak is already excluded from the feature bucket

CHECK 6 - warehouse schema, grain, and windows (gated release)


  manifest vs live schema:
  dim_clients                          9/  9 assigned  [OK]
  dim_content                         26/ 26 assigned  [OK]
  fact_content_daily_performance      31/ 31 assigned  [OK]



  iteration partition month=2026-03
    rows     : 9,841,378
    dates    : 2026-03-01 to 2026-03-31
    clients  : 55   content items: 331,437
    grain probe (report_date + client + content duplicated): 0
    [OK] warehouse grain holds


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4.1 AI traffic is sparse, but the denominator decides how sparse

Measured on the `month=2026-03` partition (9,841,378 daily rows), not quoted from the manifest:

| Measure | Value |
|---|---:|
| Daily rows | 9,841,378 |
| Rows where GA4 is available at all | 413,966 (4.21%) |
| Rows with `sessions_ai` at least 1 | 5,534 |
| ... as a share of **all** daily rows | 0.056% |
| ... as a share of **GA4-available** rows | **1.34%** |
| Content items with any AI traffic | 3,795 of 331,437 (1.15%) |
| Clients with any AI traffic | 35 |
| AI sessions as a share of all GA4 sessions | 0.686% |

**The denominator is the whole point.** Quoting "0.038% of daily rows" (the release manifest's
figure) divides by rows where 95.8% have no GA4 tracking, so AI traffic could never have been
observed there. Against the rows where it *could* be observed the rate is 1.34%, roughly 35x higher.
The honest comparison to the starter slice's 6.43% of items is the warehouse's 1.15% of items per
month, a gap of about 5x, explained by the starter being a curated slice and covering 90 days rather
than 31.

**What follows.** 3,795 positive items in a single month is a workable positive class for a ranking
task, so the AI visibility gap survives as a real objective rather than being demoted. It stays
imbalanced enough that Precision@K is the right metric and accuracy is meaningless, and the positive
rate must be reported next to every metric, because Precision@50 at a 1% base rate and Precision@50
at a 50% base rate are not comparable numbers.

### 4.2 Engine mix, which the starter dataset cannot show at all

| Engine | Sessions (Mar 2026) | Share of AI |
|---|---:|---:|
| ChatGPT | 5,155 | 57.85% |
| Gemini | 2,527 | 28.36% |
| Perplexity | 970 | 10.89% |
| Copilot | 144 | 1.62% |
| Claude | 118 | 1.32% |
| Meta, Other | 0 | 0.00% |

ChatGPT and Gemini together account for 86% of AI referrals. The starter CSV carries only a single
`ai_sessions_90d` total, so this split is invisible there. `ai_meta` and `ai_other` are entirely zero
in this partition, so they are constant columns for this month and must not be fed to a model as if
they carried information.

### 4.3 The usable client universe is 54, not 104

| access_profile | Clients | Active |
|---|---:|---:|
| gsc_and_ga4 | 53 | 41 |
| no_search_or_analytics_access | 26 | 18 |
| gsc_only | 14 | 14 |
| source_only_missing_client_dimension | 10 | 0 |
| ga4_only | 1 | 1 |

`sessions_ai` is a GA4 measure, so only `gsc_and_ga4` and `ga4_only` can contribute to this lane:
54 clients, 42 of them active. The 26 clients with `no_search_or_analytics_access` and the 10
`source_only_missing_client_dimension` rows (all inactive) are structurally excluded, not filtered
for convenience. This replaces the documentation's vague "about a third of clients have little
usable history" with a column that can be filtered on directly.

### 4.4 Zeros are not always zeros

Daily history runs 2025-01-27 to 2026-06-30, but each client's history starts when their tracking
started. Rows before a client's `ga4_data_start` carry GA4 columns zero-filled with
`ga4_data_available = FALSE`. Reading those zeros as "no engagement" would invent a decline that
never happened. Contract rule: filter on `ga4_data_available`, and build windows per client from
`gsc_data_start` and `ga4_data_start` rather than from one global calendar.

### 4.5 Window overlap in the query table

`fact_content_query_90d` covers a fixed 90-day window overlapping the snapshot's final months. If a
label lives in the last 30 days, only `*_prev30`-style columns are safe. Its per-content context
columns repeat on every row, so they need `ANY_VALUE()` and never `SUM()`, which would double count.

### 4.6 What this data can never establish

- **Causation.** No content edit is observed, so a rise after a change cannot be attributed to it
  without a controlled experiment. Everything here is observed association and decision support.
- **Why an AI engine cited a page.** Referral sessions are visible; retrieval and ranking inside
  ChatGPT, Gemini, Perplexity, Copilot or Claude are not. The per-engine split shows *that* traffic
  differs by engine, never *why*.
- **Anything about real pages, clients, or queries.** Every identifier is a pseudonym.
- **Deep seasonality for most clients.** Only 9 of 70 clients with daily facts carry 12+ months, so
  a year-over-year claim would rest on a small, self-selected subset.


In [4]:
# Section 4 - measured limits, then the contract artefact later notebooks read.
MIN_IMPRESSIONS = 500

MEASURED = {
    "partition": "month=2026-03",
    "daily_rows": 9_841_378,
    "ga4_available_rows": 413_966,
    "ai_rows": 5_534,
    "ai_content_items": 3_795,
    "content_items": 331_437,
    "ai_clients": 35,
    "ai_sessions": 8_911,
    "ga4_sessions": 1_299_808,
    "engines": {"chatgpt": 5_155, "gemini": 2_527, "perplexity": 970,
                "copilot": 144, "claude": 118, "meta": 0, "other": 0},
    "access_profile": {"gsc_and_ga4": 53, "no_search_or_analytics_access": 26,
                       "gsc_only": 14, "source_only_missing_client_dimension": 10,
                       "ga4_only": 1},
    "measured_on": "2026-08-19",
}

if warehouse_live:
    row = con.execute(
        "SELECT COUNT(*) AS n_all, "
        "SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS n_ga4, "
        "SUM(CASE WHEN sessions_ai>=1 THEN 1 ELSE 0 END) AS n_ai, "
        "COUNT(DISTINCT CASE WHEN sessions_ai>=1 THEN content_hash_id END) AS c_ai, "
        "COUNT(DISTINCT content_hash_id) AS c_all, "
        "COUNT(DISTINCT CASE WHEN sessions_ai>=1 THEN client_hash_id END) AS cl_ai, "
        "SUM(sessions_ai) AS s_ai, SUM(ga4_sessions) AS s_all "
        f"FROM '{ITER_PARTITION_FILE}'").fetchone()
    MEASURED.update(dict(zip(
        ["daily_rows", "ga4_available_rows", "ai_rows", "ai_content_items",
         "content_items", "ai_clients", "ai_sessions", "ga4_sessions"], row)))
    print("[LIVE] measured from the warehouse just now")
else:
    print(f"[RECORDED] values measured on {MEASURED['measured_on']}; set HF_TOKEN to re-measure")

d = MEASURED
print("\n" + "=" * 74)
print(f"LIMIT 4.1 - AI sparsity, and why the denominator decides ({d['partition']})")
print("=" * 74)
print(f"  daily rows                          : {d['daily_rows']:,}")
print(f"  rows with GA4 available             : {d['ga4_available_rows']:,} "
      f"({d['ga4_available_rows'] / d['daily_rows'] * 100:.2f}%)")
print(f"  rows with sessions_ai               : {d['ai_rows']:,}")
print(f"    as share of ALL daily rows        : {d['ai_rows'] / d['daily_rows'] * 100:.4f}%   <- misleading")
print(f"    as share of GA4-AVAILABLE rows    : {d['ai_rows'] / d['ga4_available_rows'] * 100:.2f}%   <- the honest one")
print(f"  content items with any AI traffic   : {d['ai_content_items']:,} of {d['content_items']:,} "
      f"({d['ai_content_items'] / d['content_items'] * 100:.2f}%)")
print(f"  clients with any AI traffic         : {d['ai_clients']}")
print(f"  AI share of all GA4 sessions        : {d['ai_sessions'] / d['ga4_sessions'] * 100:.3f}%")
print(f"\n  Starter slice for comparison        : {int((starter['ai_sessions_90d'] > 0).sum()):,} of "
      f"{len(starter):,} items ({(starter['ai_sessions_90d'] > 0).mean() * 100:.2f}%) over 90 days")
print(f"  -> {d['ai_content_items']:,} positive items in one month is a workable ranking target.")
print("     Imbalanced enough that Precision@K is the metric and accuracy is meaningless.")

print("\n" + "=" * 74)
print("LIMIT 4.2 - engine mix, invisible in the starter dataset")
print("=" * 74)
total_engine = sum(d["engines"].values())
for name, val in sorted(d["engines"].items(), key=lambda kv: -kv[1]):
    share = val / total_engine * 100 if total_engine else 0
    flag = "  <- constant this month, do not model" if val == 0 else ""
    print(f"  {name:12s} {val:>8,}  ({share:5.2f}%){flag}")

print("\n" + "=" * 74)
print("LIMIT 4.3 - usable client universe")
print("=" * 74)
ga4_capable = d["access_profile"]["gsc_and_ga4"] + d["access_profile"]["ga4_only"]
total_clients = sum(d["access_profile"].values())
for name, val in sorted(d["access_profile"].items(), key=lambda kv: -kv[1]):
    usable = "GA4-capable" if name in ("gsc_and_ga4", "ga4_only") else "excluded for this lane"
    print(f"  {name:38s} {val:4d}   {usable}")
print(f"  -> {ga4_capable} of {total_clients} clients can contribute AI signal at all.")

print("\n" + "=" * 74)
print("LIMIT 4.1b - the AI visibility gap on the starter slice")
print("=" * 74)
high_demand = starter["impressions_90d"] >= MIN_IMPRESSIONS
gap = high_demand & (starter["ai_sessions_90d"] == 0)
print(f"  high-demand items (impressions_90d at least {MIN_IMPRESSIONS}) : {int(high_demand.sum()):,} "
      f"({high_demand.mean() * 100:.2f}%)")
print(f"  of those, zero AI sessions                          : {int(gap.sum()):,} "
      f"({gap.sum() / high_demand.sum() * 100:.2f}%)")

print("\n" + "=" * 74)
print("CONTRACT ARTEFACT - written once here, read by every later notebook")
print("=" * 74)
contract = {
    "version": "1.1",
    "card": "ML-04",
    "lane": "Freestyle - AI Referral and GEO Opportunity Scoring",
    "analysis_grain": "one content item observed at a single decision date T",
    "source_grain": {
        "warehouse_fact": "report_date x client_hash_id x content_hash_id (probed, 0 duplicates)",
        "starter_csv": "content_id (probed, 0 duplicates)",
    },
    "windows": {
        "decision_date": DEV_DECISION_DATE.isoformat(),
        "feature_window": [FEATURE_WINDOW[0].isoformat(), FEATURE_WINDOW[1].isoformat()],
        "label_window": [LABEL_WINDOW[0].isoformat(), LABEL_WINDOW[1].isoformat()],
        "sealed_test_month": SEALED_TEST_MONTH,
        "iteration_partition": ITERATION_PARTITION,
    },
    "fields": {"starter": STARTER_BUCKETS, "warehouse": WAREHOUSE_BUCKETS,
               "window_dependent": WINDOW_DEPENDENT},
    "leakage_flagged": leaks,
    "labels": {
        "primary": "ga4_sessions summed over W_l below 0.80 x W_f daily average, ga4_data_available rows only",
        "secondary": f"AI visibility gap: demand in W_f at least {MIN_IMPRESSIONS} impressions and sessions_ai == 0",
        "starter_proxy": "is_declining_label = trend_direction == 'down' (current-window bucket, not a future outcome)",
    },
    "rules": {
        "split": "group by client_hash_id; never a random row split",
        "eligible_clients": "access_profile in ('gsc_and_ga4','ga4_only') for AI work",
        "missing_values": "add has_<col> indicator then impute; never a bare fillna(0)",
        "sentinels": "avg_position == 0 means no data; exclude before averaging",
        "rates": "ctr, engagement_rate, scroll_rate, ai_traffic_pct are x100 percentages",
        "query_table": "ANY_VALUE() per-content context columns; never SUM()",
        "panel": "filter on ga4_data_available; per-client windows, not one calendar",
        "constant_columns": "ai_meta and ai_other are all zero in month=2026-03; check before modelling",
        "min_impressions": MIN_IMPRESSIONS,
        "metric": "Precision@K, always reported beside the positive rate",
    },
    "measured": MEASURED,
}

out_dir = Path("work/outputs") if Path("work").is_dir() else Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "data_contract.json"
out_path.write_text(json.dumps(contract, indent=2, default=str), encoding="utf-8")

n_features = len(STARTER_BUCKETS["feature"]) + sum(
    len(b["feature"]) for b in WAREHOUSE_BUCKETS.values())
print(f"  written : {out_path.resolve()}")
print(f"  {n_features} feature fields across both datasets, {len(leaks)} leak(s) flagged")
print("\n  Later notebooks load this instead of redefining anything:")
print("      contract = json.loads(Path('work/outputs/data_contract.json').read_text())")


[LIVE] measured from the warehouse just now

LIMIT 4.1 - AI sparsity, and why the denominator decides (month=2026-03)
  daily rows                          : 9,841,378
  rows with GA4 available             : 413,966 (4.21%)
  rows with sessions_ai               : 5,534
    as share of ALL daily rows        : 0.0562%   <- misleading
    as share of GA4-AVAILABLE rows    : 1.34%   <- the honest one
  content items with any AI traffic   : 3,795 of 331,437 (1.15%)
  clients with any AI traffic         : 35
  AI share of all GA4 sessions        : 0.686%

  Starter slice for comparison        : 1,930 of 30,000 items (6.43%) over 90 days
  -> 3,795 positive items in one month is a workable ranking target.
     Imbalanced enough that Precision@K is the metric and accuracy is meaningless.

LIMIT 4.2 - engine mix, invisible in the starter dataset
  chatgpt         5,155  (57.83%)
  gemini          2,527  (28.35%)
  perplexity        970  (10.88%)
  copilot           144  ( 1.62%)
  claude       

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Note on scope.** ML-05 was retired on 2026-07-13 with its core folded into ML-04, so the leakage
test lives in section 3 of this notebook rather than in a separate one.
